# Model Building

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

RANDOM_STATE = 42


### Loading & Preparing Data

In [6]:
df = pd.read_csv('../data/Bank Customer Churn Prediction.csv')
df.columns = df.columns.str.lower()

df = df.drop(columns=['customer_id', 'surname'], errors='ignore')

df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop(columns=['churn'])
y = df_encoded['churn']

print(f"Features shape: {X.shape}, Target shape: {y.shape}")

Features shape: (10000, 11), Target shape: (10000,)


### 80/20 Train Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20, 
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape[0]} rows (Churn rate: {y_train.mean()*100:.2f}%)")
print(f"Testing set: {X_test.shape[0]} rows (Churn rate: {y_test.mean()*100:.2f}%)")

Training set: 8000 rows (Churn rate: 20.38%)
Testing set: 2000 rows (Churn rate: 20.35%)


### Model Training & Evaluation

In [8]:
models = {
    "Logistic Regression (Baseline)": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest Classifier": RandomForestClassifier(random_state=RANDOM_STATE),
    "CatBoost Classifier": CatBoostClassifier(random_state=RANDOM_STATE, verbose=0) # verbose=0 keeps output clean
}

results = {}

for name, model in models.items():
    print(f"\n--- Training {name} ---")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    results[name] = {"Accuracy": accuracy, "ROC_AUC": roc_auc}

    print(classification_report(y_test, y_pred))


--- Training Logistic Regression (Baseline) ---


/Users/java/Desktop/Aliya's project/Churn-Prediction-Bank-Customer-Retention-Strategy/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

           0       0.82      0.97      0.89      1593
           1       0.60      0.18      0.28       407

    accuracy                           0.81      2000
   macro avg       0.71      0.58      0.59      2000
weighted avg       0.78      0.81      0.77      2000


--- Training Random Forest Classifier ---
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1593
           1       0.78      0.46      0.58       407

    accuracy                           0.86      2000
   macro avg       0.83      0.71      0.75      2000
weighted avg       0.86      0.86      0.85      2000


--- Training CatBoost Classifier ---
              precision    recall  f1-score   support

           0       0.88      0.97      0.92      1593
           1       0.79      0.49      0.61       407

    accuracy                           0.87      2000
   macro avg       0.84      0.73      0.76    

### Comparing Model Performance

In [13]:
comparison_df = pd.DataFrame(results).T

print("Raw Model Metrics Preview")
print(comparison_df)
print("\nDataFrame Columns Available:", list(comparison_df.columns))

possible_auc_keys = ['ROC-AUC', 'roc-auc', 'Roc-Auc', 'AUC', 'auc']
found_key = next((key for key in possible_auc_keys if key in comparison_df.columns), None)

print("\nSorted Model Performance Comparison")
if found_key:
    print(comparison_df.sort_values(by=found_key, ascending=False))
else:
    print(comparison_df.sort_values(by=comparison_df.columns[0], ascending=False))

Raw Model Metrics Preview
                                Accuracy   ROC_AUC
Logistic Regression (Baseline)    0.8085  0.757334
Random Forest Classifier          0.8645  0.852629
CatBoost Classifier               0.8700  0.862173

DataFrame Columns Available: ['Accuracy', 'ROC_AUC']

Sorted Model Performance Comparison
                                Accuracy   ROC_AUC
CatBoost Classifier               0.8700  0.862173
Random Forest Classifier          0.8645  0.852629
Logistic Regression (Baseline)    0.8085  0.757334
